In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from __future__ import print_function

import os
import keras
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.preprocessing import image_dataset_from_directory
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.vgg19 import VGG19, preprocess_input

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.utils import plot_model
%ls

In [ ]:
batch_size = 32  
epochs = 11
loss = tf.keras.losses.CategoricalCrossentropy()
metrics = [
      tf.keras.metrics.BinaryAccuracy(name='accuracy'),
      tf.keras.metrics.Precision(name='precision'),
      tf.keras.metrics.Recall(name='recall'),  
      tf.keras.metrics.AUC(name='auc')]

In [ ]:
input_size = (224,224)

train_datagen = ImageDataGenerator(rescale = 1./255,
                                   validation_split = 0.2,                                  
                                   rotation_range=5,
                                   width_shift_range=0.2,
                                   height_shift_range=0.1,
                                   shear_range=0.2,
                                   horizontal_flip=True,
                                   vertical_flip=True,
                                   fill_mode="nearest")

valid_datagen = ImageDataGenerator(rescale = 1./255,
                                  validation_split = 0.1)

test_datagen  = ImageDataGenerator(rescale = 1./255,
                                   validation_split=0.1
                                  )

In [ ]:
train_data = train_datagen.flow_from_directory(directory = "/content/drive/MyDrive/BIYOINFORMATIKVERISETIORNEKLER",
                                                target_size = input_size, 
                                                class_mode = "categorical",
                                                subset = "training",
                                                batch_size = batch_size)

valid_data = train_datagen.flow_from_directory(directory = "/content/drive/MyDrive/BIYOINFORMATIKVERISETIORNEKLER",
                                                target_size = input_size,
                                                class_mode = "categorical",
                                                subset = "validation",
                                                batch_size = batch_size)

test_data = test_datagen.flow_from_directory(directory = "/content/drive/MyDrive/BIYOINFORMATIKVERISETIORNEKLER",
                                                target_size = input_size,
                                                class_mode = "categorical",
                                                batch_size = batch_size)

In [ ]:
def exponential_decay(lr0, s):
    def exponential_decay_fn(epoch):
        return lr0 * 0.1 **(epoch / s)
    return exponential_decay_fn

exponential_decay_fn = exponential_decay(0.01, 5)

callbacks = tf.keras.callbacks.LearningRateScheduler(exponential_decay_fn)

**VGG NET**

In [ ]:
model_vgg19 = VGG19(input_shape = (224,224,3), include_top=False, weights="imagenet")

In [ ]:
for each_layer in model_vgg19.layers:
    each_layer.trainable = False

In [ ]:
import tensorflow
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import InputLayer, BatchNormalization, Dropout, Flatten, Dense, Activation
from keras import layers

In [ ]:
model_2=Sequential()

model_2.add(model_vgg19)
model_2.add(Dropout(0.5))
model_2.add(Flatten())
model_2.add(BatchNormalization())

model_2.add(Dense(64,kernel_initializer='he_uniform'))
model_2.add(BatchNormalization())
model_2.add(Activation('relu'))
model_2.add(Dropout(0.5))

model_2.add(Dense(64,kernel_initializer='he_uniform'))
model_2.add(BatchNormalization())
model_2.add(Activation('relu'))
model_2.add(Dropout(0.5))

model_2.add(Dense(64,kernel_initializer='he_uniform'))
model_2.add(BatchNormalization())
model_2.add(Activation('relu'))
model_2.add(Dropout(0.5))

model_2.add(Dense(32,kernel_initializer='he_uniform'))
model_2.add(BatchNormalization())
model_2.add(Activation('relu'))
model_2.add(Dropout(0.5))

model_2.add(Dense(32,kernel_initializer='he_uniform'))
model_2.add(BatchNormalization())
model_2.add(Activation('relu'))
model_2.add(Dense(5,activation='softmax'))

In [ ]:
model_2.summary()

In [ ]:
model_2.compile(optimizer='rmsprop', 
                loss='categorical_crossentropy',
                metrics=metrics)

In [ ]:
history_2 = model_2.fit(train_data,
                  validation_data=valid_data,
                  epochs = epochs,
                  verbose = 1,
                  callbacks = callbacks)

In [ ]:
model_2.save('models/wthvgg19.h5')

In [ ]:
plt.plot(range(1, len(history_2.history['accuracy']) + 1), history_2.history['accuracy'])
plt.plot(range(1, len(history_2.history['val_accuracy']) + 1), history_2.history['val_accuracy'])
plt.title("VGG")
plt.xlabel("epochs")
plt.ylabel("accuracy")
plt.legend(['training', 'validation'])

In [ ]:
plt.plot(range(1, len(history_2.history['loss']) + 1), history_2.history['loss'])
plt.plot(range(1, len(history_2.history['val_loss']) + 1), history_2.history['val_loss'])
plt.title("VGG")
plt.xlabel("epochs")
plt.ylabel("loss")
plt.legend(['training', 'validation'])

In [ ]:
scores_2 = model_2.evaluate_generator(test_data)

In [ ]:
print("Accuracy:", scores_2[1])
print("Precision:", scores_2[2])
print("Recall:", scores_2[3])
print("AUC:", scores_2[4])